# 00.3 — Temporal Integrity Re-Audit & Canonical Split

## 1. Mục tiêu
Sau khi áp dụng quyết định loại bỏ năm 2000, **bắt buộc phải chạy lại toàn bộ Quality Re-Audit** trên dataset mới (2001–2026):
1. **Kích thước Dataset mới**: $9.251$ dòng $\times$ $36$ cột.
2. **Kiểm tra Missing**: Phải đạt đúng **$0$ ô thiếu ($0.0\%$)** trên toàn bộ 36 cột.
3. **Kiểm tra Tính liên tục Thời gian Tuyệt đối (Temporal Continuity)**: Xác nhận không có khoảng trống ngày (0 date gaps) từ `2001-01-01` đến `2026-04-30`.
4. **Tính lại Canonical Split (Recompute Canonical Split)**:
   - Điểm cắt (Cutoff): `2020-05-01`.
   - **Tập Train**: `2001-01-01` đến `2020-04-30` $\to$ **$7.060$ ngày** (~$76.3\%$).
   - **Tập Test**: `2020-05-01` đến `2026-04-30` $\to$ **$2.191$ ngày** (~$23.7\%$).
   - Tổng cộng: **$9.251$ ngày**.
5. **Xuất Artifact**: `outputs/canonical_split_report.json`.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import DataLoader, time_series_split
from src.data.dataquality import check_continuity

# 1. Nạp dataset sạch sau khi loại bỏ năm 2000 (exclude_year_2000=True, drop_redundant_radiation=False)
loader = DataLoader()
df_clean = loader.load_data(exclude_year_2000=True, drop_redundant_radiation=False)

print("=" * 70)
print("🔍 RE-AUDIT TOÀN DIỆN DỮ LIỆU SAU KHI LOẠI BỎ NĂM 2000")
print("=" * 70)
print(f"Kích thước dataset sạch: {df_clean.shape[0]:,} dòng x {df_clean.shape[1]} cột")
assert df_clean.shape == (9251, 36), f"Expected (9251, 36), got {df_clean.shape}"

# 2. Assertions Chất lượng Dữ liệu Bắt buộc
missing_total = int(df_clean.isna().sum().sum())
dup_total = int(df_clean.duplicated().sum())
print(f"   • Tổng số ô missing: {missing_total}")
print(f"   • Số dòng duplicate: {dup_total}")
assert missing_total == 0, f"Expected 0 missing cells, got {missing_total}"
assert dup_total == 0, f"Expected 0 duplicate rows, got {dup_total}"

# 3. Kiểm định Tính liên tục Thời gian Tuyệt đối (Temporal Continuity)
is_continuous, missing_dates = check_continuity(df_clean, date_col="Ngày", raise_on_gap=True)
print(f"   • Dữ liệu liên tục theo ngày: {is_continuous}")
print(f"   • Số ngày bị khuyết (date gaps): {len(missing_dates)}")
print(f"   • Dải ngày hoàn chỉnh: {df_clean['Ngày'].min().date()} → {df_clean['Ngày'].max().date()}")
assert is_continuous == True, "Date gaps detected in clean dataset!"
assert len(missing_dates) == 0, "Missing dates found!"


🔍 RE-AUDIT TOÀN DIỆN DỮ LIỆU SAU KHI LOẠI BỎ NĂM 2000
Kích thước dataset sạch: 9,251 dòng x 36 cột
   • Tổng số ô missing: 0
   • Số dòng duplicate: 0
   • Dữ liệu liên tục theo ngày: True
   • Số ngày bị khuyết (date gaps): 0
   • Dải ngày hoàn chỉnh: 2001-01-01 → 2026-04-30


In [2]:
# 4. Tính toán lại Canonical Split (Recompute Canonical Split)
train_df, test_df = time_series_split(df_clean, date_col="Ngày")

train_count = len(train_df)
test_count = len(test_df)
total_count = len(df_clean)

print("\n" + "=" * 70)
print("📊 CANONICAL TIME SERIES SPLIT (RECOMPUTED):")
print("=" * 70)
print(f"   • Train Split: {train_count:,} ngày ({train_count/total_count:.1%}) [{train_df['Ngày'].min().date()} → {train_df['Ngày'].max().date()}]")
print(f"   • Test Split:  {test_count:,} ngày ({test_count/total_count:.1%}) [{test_df['Ngày'].min().date()} → {test_df['Ngày'].max().date()}] [QUARANTINED]")
print(f"   • Mốc ranh giới (Cutoff Date): 2020-05-01")
print(f"   • Tổng số ngày: {total_count:,} ngày")

# Assertions Canonical Split
assert train_count == 7060, f"Expected 7,060 train days, got {train_count}"
assert test_count == 2191, f"Expected 2,191 test days, got {test_count}"
assert train_df['Ngày'].max() < test_df['Ngày'].min(), "Train and Test overlap detected!"

# 5. Xuất Canonical Split Artifact
OUTPUT_DIR = ROOT_DIR / "notebooks" / "data_quality" / "outputs"
canonical_artifact = {
    "clean_dataset_rows": total_count,
    "clean_dataset_columns": df_clean.shape[1],
    "train_split": {
        "start_date": str(train_df["Ngày"].min().date()),
        "end_date": str(train_df["Ngày"].max().date()),
        "days_count": train_count,
        "percentage": round(train_count / total_count * 100, 2)
    },
    "test_split": {
        "start_date": str(test_df["Ngày"].min().date()),
        "end_date": str(test_df["Ngày"].max().date()),
        "days_count": test_count,
        "percentage": round(test_count / total_count * 100, 2)
    },
    "cutoff_date": "2020-05-01",
    "leakage_invariants": "Train split is strictly frozen for all EDA statistics and model transformations. Test split is quarantined."
}

artifact_path = OUTPUT_DIR / "canonical_split_report.json"
with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(canonical_artifact, f, indent=2, ensure_ascii=False)

print(f"\n💾 Canonical split report exported to: {artifact_path}")



📊 CANONICAL TIME SERIES SPLIT (RECOMPUTED):
   • Train Split: 7,060 ngày (76.3%) [2001-01-01 → 2020-04-30]
   • Test Split:  2,191 ngày (23.7%) [2020-05-01 → 2026-04-30] [QUARANTINED]
   • Mốc ranh giới (Cutoff Date): 2020-05-01
   • Tổng số ngày: 9,251 ngày

💾 Canonical split report exported to: D:\DS108_project\notebooks\data_quality\outputs\canonical_split_report.json
